# 🚀 md-editor 端侧小模型: Google Colab L4/T4 极速微调与发布

本 Notebook 可在 **Google Colab (推荐 L4 / T4 GPU)** 上一键完成：
1. **环境自检**：检测 NVIDIA L4 / T4 GPU 与硬件加速状态
2. **高吞吐微调**：运行 RFC-002 SFT 微调（L4 GPU 上 0.5B 仅需 **5~7 分钟**！）
3. **自动量化**：通过 `llama.cpp` 转换为 `Q4_K_M` GGUF 格式
4. **自动发布**：自动发布至 GitHub Releases，一键生成 Manifest 描述文件

In [ ]:
#@title ⚙️ [1/4] 配置训练参数与基座选择
#@markdown 请在右侧面板选择你要训练的模型规格：

model_tier = "0.5B (Lite - L4约5分钟)" #@param ["0.5B (Lite - L4约5分钟)", "1.5B (Standard - L4约15分钟)"]
version_tag = "v1.0.0" #@param {type:"string"}

if "1.5B" in model_tier:
    BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
else:
    BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"🎯 已选择基座: {BASE_MODEL}")
print(f"🏷️ 版本标签:   {version_tag}")

# 验证 GPU 状态 (需显示 L4 / T4)
!nvidia-smi

In [ ]:
#@title 📦 [2/4] 克隆/更新仓库并预装全套依赖环境
import os
if not os.path.exists('/content/md-editor-models'):
    !git clone https://github.com/wmasfoe/md-editor-models.git /content/md-editor-models
else:
    !git -C /content/md-editor-models pull origin master

%cd /content/md-editor-models
!git pull origin master
!pip install -q trl peft pangu datasets transformers accelerate sentencepiece gguf protobuf

In [ ]:
#@title 🔑 [3/4] 配置 GitHub Token (用于自动发布 Release)
import os
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception:
    token = None

if not token and not os.environ.get('GH_TOKEN'):
    token = input("请输入你的 GitHub Token (按回车直接上传): ").strip()

if token:
    os.environ['GH_TOKEN'] = token
    os.environ['GITHUB_TOKEN'] = token
    print("✅ GitHub Token 配置成功！")
else:
    print("ℹ️ 未提供 Token，训练完成后模型将保存在 output 目录。")

In [ ]:
#@title 🚀 [4/4] 启动一键极速微调、量化与 GitHub Release 发布！
!git pull origin master
!chmod +x scripts/release_model.sh
!./scripts/release_model.sh $version_tag $BASE_MODEL

print("\n🎉 全流程执行完毕！")